# NeuroVLM PubMed Semantic Evaluation Baseline

This notebook evaluates the pretrained NeuroVLM MLP baseline on the official PubMed split. Exact PubMed paper retrieval is kept as `exact_pmid_retrieval`, but semantic ranking is the main comparison target: network labels, PubMed MeSH terms, and semantic-neighborhood paper retrieval.

Run this first and keep the output directory as the MLP baseline reference when comparing ALE3DCNN, CoordGNN, CoordDeepSet, ALEFlatMLP, and DiFuMo GAT runs.

## 1. Imports / Device

In [1]:
from pathlib import Path
import sys
import json
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

repo_root = Path.cwd()
if (repo_root / "src").exists():
    sys.path.insert(0, str(repo_root / "src"))

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
    else "cpu"
)
print("device:", device)
print("torch:", torch.__version__)

device: mps
torch: 2.10.0


## 2. Config

In [2]:
from neurovlm.semantic_evaluation import find_default_mesh_json

OUT_DIR = Path("runs/neurovlm_pubmed_semantic_baseline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

KS = (1, 5, 10, 50)
BATCH_SIZE = 4096
TEXT_EMBED_BATCH_SIZE = 64
RANDOM_SPLIT_SEED = 0

RUN_NETWORK_LABELING = True
RUN_MESH_TERM_RANKING = True
RUN_SEMANTIC_NEIGHBOR_RETRIEVAL = True

NETWORK_LABELS_CSV = Path("experiments/data/networks_labels/network_test_set_labels.csv")
MESH_PMID_JSON = find_default_mesh_json()  # set this manually if your PMID->MeSH JSON lives elsewhere
MESH_CANDIDATES_FROM_ALL_PMIDS = True
SEMANTIC_NEIGHBORS = 10

print("out_dir:", OUT_DIR.resolve())
print("network_labels_csv:", NETWORK_LABELS_CSV if NETWORK_LABELS_CSV.exists() else "auto-detect")
print("mesh_pmid_json:", MESH_PMID_JSON)

out_dir: /Users/borng/code/lab_work/neurovlm/experiments/runs/neurovlm_pubmed_semantic_baseline
network_labels_csv: auto-detect
mesh_pmid_json: data/mesh_kg/mesh_annotations.json


## 3. Load PubMed Latents, Metadata, And Model

In [3]:
from neurovlm.core import NeuroVLM
from neurovlm.data import load_dataset
from neurovlm.retrieval_resources import (
    _load_latent_neuro,
    _load_latent_text,
    _proj_head_text_infonce,
)

brain_latent, brain_pmids = _load_latent_neuro()
text_latent, text_pmids = _load_latent_text()
pub_df = load_dataset("pubmed_text").copy()

brain_pmids = np.asarray(brain_pmids).astype(str)
text_pmids = np.asarray(text_pmids).astype(str)
pub_df["pmid"] = pub_df["pmid"].astype(str)

nvlm = NeuroVLM(datasets=["pubmed"], device=str(device))
proj_head_text = _proj_head_text_infonce().to(device).eval()

print("brain_latent:", tuple(brain_latent.shape), "brain_pmids:", len(brain_pmids))
print("text_latent :", tuple(text_latent.shape), "text_pmids :", len(text_pmids))
print("publications:", pub_df.shape, "columns:", list(pub_df.columns))
print("NeuroVLM datasets:", nvlm.datasets)

brain_latent: (29868, 384) brain_pmids: 29868
text_latent : (30826, 768) text_pmids : 30826
publications: (30826, 8) columns: ['pmid', 'pmcid', 'doi', 'name', 'description', 'train', 'test', 'val']
NeuroVLM datasets: ('pubmed',)


## 4. Align Brain/Text Rows By PMID

In [4]:
brain_lookup = {p: i for i, p in enumerate(brain_pmids)}
brain_rows, text_rows, shared_pmids = [], [], []
for text_i, pmid in enumerate(text_pmids):
    brain_i = brain_lookup.get(pmid)
    if brain_i is None:
        continue
    brain_rows.append(brain_i)
    text_rows.append(text_i)
    shared_pmids.append(pmid)

brain_rows = np.asarray(brain_rows, dtype=np.int64)
text_rows = np.asarray(text_rows, dtype=np.int64)
shared_pmids = np.asarray(shared_pmids).astype(str)
assert np.array_equal(brain_pmids[brain_rows], text_pmids[text_rows])

print("aligned pairs:", len(shared_pmids))
print("first 5 alignment checks:")
for i in range(min(5, len(shared_pmids))):
    print(i, "pmid=", shared_pmids[i], "brain_pmid=", brain_pmids[brain_rows[i]], "text_pmid=", text_pmids[text_rows[i]])

aligned pairs: 29868
first 5 alignment checks:
0 pmid= 1589767 brain_pmid= 1589767 text_pmid= 1589767
1 pmid= 8530552 brain_pmid= 8530552 text_pmid= 8530552
2 pmid= 8624678 brain_pmid= 8624678 text_pmid= 8624678
3 pmid= 8670634 brain_pmid= 8670634 text_pmid= 8670634
4 pmid= 8994101 brain_pmid= 8994101 text_pmid= 8994101


## 5. Use Official PubMed Split Columns

In [5]:
from neurovlm.semantic_evaluation import save_official_pubmed_splits

splits = save_official_pubmed_splits(
    pub_df,
    shared_pmids,
    OUT_DIR,
    random_state=RANDOM_SPLIT_SEED,
)

test_pmid_set = set(splits["test"].astype(str).tolist())
val_pmid_set = set(splits["val"].astype(str).tolist())
train_pmid_set = set(splits["train"].astype(str).tolist())

test_pos = np.flatnonzero(np.asarray([pmid in test_pmid_set for pmid in shared_pmids]))
val_pos = np.flatnonzero(np.asarray([pmid in val_pmid_set for pmid in shared_pmids]))
train_pos = np.flatnonzero(np.asarray([pmid in train_pmid_set for pmid in shared_pmids]))

assert len(test_pos) == len(splits["test"]), "Some official test PMIDs did not align to both brain and text latents."
print({"train": len(train_pos), "val": len(val_pos), "test": len(test_pos)})

PubMed split counts: {'train': 23894, 'val': 2987, 'test': 2987}
{'train': 23894, 'val': 2987, 'test': 2987}


## 6. Exact PMID Retrieval Diagnostic

In [6]:
from neurovlm.semantic_evaluation import exact_pmid_retrieval_metrics, exact_recall_curve

# Query = PubMed brain latent; candidates = exact PubMed papers from the same official test split.
test_brain_raw = brain_latent[brain_rows[test_pos]].float()
test_pmids = shared_pmids[test_pos]
candidate_text_rows = text_rows[test_pos]
candidate_pmids = text_pmids[candidate_text_rows]
assert np.array_equal(test_pmids, candidate_pmids)

result = nvlm.to_text(test_brain_raw, datasets=["pubmed"], project=True)
scores_all = result.scores_by_dataset["pubmed"]
sim = scores_all[candidate_text_rows].T.contiguous()

brain_shared = result.query_embeddings.float()
with torch.no_grad():
    text_shared = F.normalize(
        proj_head_text(text_latent[candidate_text_rows].float().to(device)).detach().cpu(),
        dim=1,
        eps=1e-8,
    )

exact_b2p = exact_pmid_retrieval_metrics(sim, ks=KS)
exact_p2b = exact_pmid_retrieval_metrics(sim.T, ks=KS)
exact_summary = {
    "n_test": int(len(test_pmids)),
    "brain_to_paper": exact_b2p,
    "paper_to_brain": exact_p2b,
    "interpretation": "exact_pmid_retrieval is a strict diagnostic, not the main success metric.",
}

pd.DataFrame([{"direction": "brain_to_paper", **exact_b2p}, {"direction": "paper_to_brain", **exact_p2b}])

,direction,paper_recall_curve_auc,mrr,median_rank,mean_rank,recall@1,random_recall@1,recall@5,random_recall@5,recall@10,random_recall@10,recall@50,random_recall@50
0,brain_to_paper,0.833709,0.039668,217.0,497.709747,0.010378,0.000335,0.051892,0.001674,0.093070,0.003348,0.241379,0.016739
1,paper_to_brain,0.836605,0.042355,230.0,489.059601,0.015065,0.000335,0.051557,0.001674,0.086374,0.003348,0.239036,0.016739


In [7]:
recall_curves = pd.DataFrame({
    "k": np.arange(1, len(test_pmids) + 1),
    "brain_to_paper_recall_curve": exact_recall_curve(sim).cpu().numpy(),
    "paper_to_brain_recall_curve": exact_recall_curve(sim.T).cpu().numpy(),
    "random_recall_curve": np.arange(1, len(test_pmids) + 1) / float(len(test_pmids)),
})

with open(OUT_DIR / "exact_pmid_retrieval_metrics.json", "w") as f:
    json.dump(exact_summary, f, indent=2)
recall_curves.to_csv(OUT_DIR / "exact_pmid_retrieval_curves.csv", index=False)

# Backward-compatible aliases for older notes/scripts that expected these filenames.
with open(OUT_DIR / "pubmed_test_recall_metrics.json", "w") as f:
    json.dump(exact_summary, f, indent=2)
recall_curves.to_csv(OUT_DIR / "pubmed_test_recall_curves.csv", index=False)
print("saved exact PMID retrieval artifacts to", OUT_DIR.resolve())

saved exact PMID retrieval artifacts to /Users/borng/code/lab_work/neurovlm/experiments/runs/neurovlm_pubmed_semantic_baseline


## 7. Exact Retrieval Rank Inspection

In [8]:
from neurovlm.metrics import retrieval_ranks

ranks = retrieval_ranks(sim).cpu().numpy()
rank_df = pd.DataFrame({
    "pmid": test_pmids,
    "rank": ranks,
    "hit@10": ranks <= 10,
    "true_score": sim.diag().cpu().numpy(),
    "top1_pos": sim.argmax(dim=1).cpu().numpy(),
    "top1_score": sim.max(dim=1).values.cpu().numpy(),
})
rank_df["top1_pmid"] = test_pmids[rank_df["top1_pos"].to_numpy()]
rank_df.to_csv(OUT_DIR / "exact_pmid_retrieval_ranks.csv", index=False)
rank_df.to_csv(OUT_DIR / "pubmed_test_brain_to_paper_ranks.csv", index=False)

display(rank_df.sort_values("rank").head(10))
display(rank_df.sort_values("rank", ascending=False).head(10))

,pmid,rank,hit@10,true_score,top1_pos,top1_score,top1_pmid
1381,25061565,1,True,0.358555,1381,0.358555,25061565
2482,33414733,1,True,0.411015,2482,0.411015,33414733
598,20204074,1,True,0.388027,598,0.388027,20204074
2176,30798012,1,True,0.353785,2176,0.353785,30798012
271,17126370,1,True,0.405632,271,0.405632,17126370
469,19240659,1,True,0.364428,469,0.364428,19240659
1014,23046904,1,True,0.410907,1014,0.410907,23046904
2445,33192481,1,True,0.350599,2445,0.350599,33192481
23,11230097,1,True,0.395947,23,0.395947,11230097
1006,22993433,1,True,0.421701,1006,0.421701,22993433


,pmid,rank,hit@10,true_score,top1_pos,top1_score,top1_pmid
1936,28782545,2968,False,-0.202803,486,0.385454,19400679
2953,38342187,2955,False,-0.198220,1933,0.417467,28760866
76,13010008,2921,False,-0.205250,720,0.487716,21289179
2171,30735675,2913,False,-0.072319,2402,0.318916,32664021
92,14704218,2870,False,-0.053094,2908,0.319408,37743995
123,15294149,2856,False,-0.085480,525,0.337610,19646537
1694,26961091,2855,False,-0.083369,1736,0.431670,27262510
2839,36914701,2845,False,-0.205786,789,0.368377,21718759
49,12457757,2840,False,-0.146615,2257,0.408359,31439831
2029,29501856,2835,False,-0.118928,1263,0.341592,24375687


## 8. Network Labeling Evaluation

In [9]:
if RUN_NETWORK_LABELING:
    from neurovlm.models import Specter
    from neurovlm.retrieval_resources import _load_autoencoder, _load_masker, _proj_head_image_infonce
    from neurovlm.semantic_evaluation import (
        align_network_ground_truth,
        build_network_label_corpus,
        encode_network_maps,
        encode_texts_with_specter,
        evaluate_network_labeling,
        load_network_label_table,
        load_network_maps,
        preprocess_network_maps,
    )

    specter = Specter("allenai/specter2_aug2023refresh", adapter="adhoc_query", device=str(device))
    masker = _load_masker()
    autoencoder = _load_autoencoder()
    proj_head_image = _proj_head_image_infonce()

    labels_df = load_network_label_table(NETWORK_LABELS_CSV if NETWORK_LABELS_CSV.exists() else None)
    label_corpus = build_network_label_corpus(labels_df)
    label_embeddings = encode_texts_with_specter(
        label_corpus["text"].tolist(),
        specter,
        proj_head_text,
        batch_size=TEXT_EMBED_BATCH_SIZE,
        device=device,
    )

    network_records = load_network_maps()
    networks_resampled = preprocess_network_maps(network_records, masker)
    network_embeddings = encode_network_maps(
        "neurovlm_mlp",
        networks_resampled,
        {"masker": masker, "autoencoder": autoencoder, "proj_head_image": proj_head_image},
        device=device,
    )
    truth_df = align_network_ground_truth(networks_resampled, labels_df)
    network_metrics, network_predictions = evaluate_network_labeling(
        network_embeddings,
        label_embeddings,
        truth_df,
        label_corpus,
        out_dir=OUT_DIR,
    )
    display(pd.DataFrame([network_metrics]).drop(columns=["classification_report", "one_vs_rest_auc"], errors="ignore"))
    display(network_predictions.head(20))
else:
    print("RUN_NETWORK_LABELING is False; skipping network labeling.")

There are adapters available but none are activated for the forward pass.
Encoding network maps: 100%|██████████| 185/185 [00:23<00:00,  7.90it/s]


,n_network_maps,n_evaluated,n_skipped_unknown,accuracy,top_2_accuracy,top_3_accuracy,macro_auc
0,185,119,66,0.789916,0.92437,0.941176,0.962841


,row_index,atlas,raw_network_label,network_key,network_name,predicted_network_key,predicted_network_name,top_1_network_key,top_1_network_name,top_1_score,top_2_network_key,top_2_network_name,top_2_score,top_3_network_key,top_3_network_name,top_3_score
0,1,Du,CG-OP,cingulo_opercular,Cingulo-Opercular,motor,Motor,motor,Motor,0.201384,cingulo_opercular,Cingulo-Opercular,0.197596,frontoparietal_control,Frontoparietal Control,0.052027
1,2,Du,DN-B,default_mode,Default Mode,default_mode,Default Mode,default_mode,Default Mode,0.333122,language,Language,0.142258,frontoparietal_control,Frontoparietal Control,0.010677
2,3,Du,SMOT-B,motor,Motor,motor,Motor,motor,Motor,0.351288,auditory,Auditory,0.153786,language,Language,-0.032218
3,4,Du,AUD,auditory,Auditory,auditory,Auditory,auditory,Auditory,0.525661,language,Language,0.180555,motor,Motor,-0.030445
4,5,Du,PM-PPr,motor,Motor,motor,Motor,motor,Motor,0.241096,attention,Attention,0.202232,frontoparietal_control,Frontoparietal Control,0.079855
5,6,Du,dATN-B,attention,Attention,visual,Visual,visual,Visual,0.461924,attention,Attention,0.267278,language,Language,-0.030141
6,7,Du,SMOT-A,motor,Motor,motor,Motor,motor,Motor,0.471273,auditory,Auditory,-0.018255,cingulo_opercular,Cingulo-Opercular,-0.063899
7,8,Du,LANG,language,Language,language,Language,language,Language,0.364119,auditory,Auditory,0.239533,default_mode,Default Mode,0.035172
8,9,Du,FPN-B,frontoparietal_control,Frontoparietal Control,frontoparietal_control,Frontoparietal Control,frontoparietal_control,Frontoparietal Control,0.200437,default_mode,Default Mode,0.188465,attention,Attention,0.078462
9,10,Du,FPN-A,frontoparietal_control,Frontoparietal Control,frontoparietal_control,Frontoparietal Control,frontoparietal_control,Frontoparietal Control,0.341216,attention,Attention,0.240866,cingulo_opercular,Cingulo-Opercular,0.211780


## 9. PubMed MeSH Term Ranking

In [10]:
if RUN_MESH_TERM_RANKING:
    from neurovlm.models import Specter
    from neurovlm.semantic_evaluation import (
        build_mesh_candidate_corpus,
        definition_lookup_from_dataframe,
        encode_texts_with_specter,
        evaluate_mesh_term_ranking,
        load_pmid_mesh_map,
    )

    if MESH_PMID_JSON is None:
        raise FileNotFoundError("Set MESH_PMID_JSON to your PMID -> MeSH terms JSON path.")

    pmid_mesh = load_pmid_mesh_map(MESH_PMID_JSON)
    print("PMID->MeSH map:", MESH_PMID_JSON, "PMIDs:", len(pmid_mesh))

    definition_lookup = {}
    try:
        from neurovlm.retrieval_resources import _load_kg_mesh_dataset
        mesh_defs = _load_kg_mesh_dataset()
        definition_lookup = definition_lookup_from_dataframe(mesh_defs)
        print("Loaded MeSH definitions:", len(definition_lookup))
    except Exception as exc:
        warnings.warn(f"Could not load MeSH definition dataframe; using term-only embeddings where needed: {exc}")

    candidate_pmids = list(pmid_mesh.keys()) if MESH_CANDIDATES_FROM_ALL_PMIDS else test_pmids
    mesh_corpus = build_mesh_candidate_corpus(
        pmid_mesh,
        candidate_pmids,
        definition_lookup=definition_lookup,
        strip_qualifiers_for_candidates=True,
    )
    print("MeSH candidate terms:", len(mesh_corpus))

    if "specter" not in globals():
        specter = Specter("allenai/specter2_aug2023refresh", adapter="adhoc_query", device=str(device))
    mesh_embeddings = encode_texts_with_specter(
        mesh_corpus["text"].tolist(),
        specter,
        proj_head_text,
        batch_size=TEXT_EMBED_BATCH_SIZE,
        device=device,
    )

    mesh_metrics, mesh_predictions = evaluate_mesh_term_ranking(
        brain_shared,
        test_pmids,
        mesh_embeddings,
        mesh_corpus,
        pmid_mesh,
        out_dir=OUT_DIR,
    )
    display(pd.DataFrame([mesh_metrics]))
    display(mesh_predictions.head(20))
else:
    print("RUN_MESH_TERM_RANKING is False; skipping MeSH term ranking.")

PMID->MeSH map: data/mesh_kg/mesh_annotations.json PMIDs: 1231613
Loaded MeSH definitions: 64128
MeSH candidate terms: 26503


Encoding label texts: 100%|██████████| 415/415 [01:55<00:00,  3.58it/s]


,recall@1,recall@5,recall@10,recall@50,map,mrr,ndcg@10,median_best_true_term_rank,average_true_term_rank,n_queries,n_candidates
0,0.031008,0.104855,0.161159,0.330477,0.009694,0.074332,0.022326,167.0,10909.583391,2451,26503


,pmid,true_mesh_terms,top_20_predicted_terms,scores,hit@5,hit@10,best_true_term_rank
0,8994101,Acoustic Impedance Tests|Adult|Aged|Audiometry...,Surgical Equipment|Equipment and Supplies|Anes...,0.477160|0.474896|0.474119|0.469546|0.466524|0...,False,False,870.0
1,9065511,Adult|Brain|Brain Mapping|Environment|Female|H...,"Contactins|Orientation, Spatial|Spatial Proces...",0.284106|0.278991|0.275894|0.272455|0.272435|0...,False,False,21.0
2,9084599,Adult|Attention|Cerebrovascular Circulation|Fi...,Spinocerebellar Tracts|Efferent Pathways|Spino...,0.332215|0.309947|0.308309|0.298058|0.294680|0...,False,False,381.0
3,10202567,Analysis of Variance|Auditory Perception|Blood...,"Books, Illustrated|Candy|Starlings|Jupiter|Ton...",0.271452|0.214891|0.202291|0.199353|0.195196|0...,False,False,188.0
4,10220229,Adult|Female|Humans|Magnetic Resonance Imaging...,"Cues|Genes, APC|Psychology, Military|Prefronta...",0.264190|0.223579|0.218337|0.213597|0.213490|0...,True,True,4.0
5,10401985,Adult|Analysis of Variance|Attention|Brain Map...,"Saccades|Reaction Time|Computers, Mainframe|Di...",0.341900|0.317175|0.306531|0.301431|0.300321|0...,False,False,29.0
6,10426416,Adult|Algorithms|Brain|Brain Mapping|Cerebral ...,"Air|Olfactory Mucosa|Sleep, REM|Limbic System|...",0.211133|0.195966|0.194493|0.192202|0.191723|0...,False,False,681.0
7,10450889,,Form Perception|Face|Aphakia|Figural Aftereffe...,0.428647|0.426653|0.423538|0.416184|0.406957|0...,False,False,NaN
8,10586971,Adult|Brain Mapping|Female|Humans|Magnetic Res...,"Dominance, Cerebral|Multitasking Behavior|Func...",0.346306|0.336928|0.331070|0.324784|0.323852|0...,False,False,16.0
9,10607401,Adult|Brain Mapping|Culture|England|Frontal Lo...,Psycholinguistics|Vocabulary|Dyslexia|Writing|...,0.439653|0.431654|0.430867|0.425666|0.411288|0...,True,True,5.0


## 10. Semantic-Neighborhood Paper Retrieval

In [15]:
if RUN_SEMANTIC_NEIGHBOR_RETRIEVAL:
    from neurovlm.semantic_evaluation import evaluate_semantic_neighbor_retrieval

    semantic_metrics, semantic_examples = evaluate_semantic_neighbor_retrieval(
        brain_shared,
        text_shared,
        test_pmids,
        n_neighbors=SEMANTIC_NEIGHBORS,
        out_dir=OUT_DIR,
    )
    display(pd.DataFrame([semantic_metrics]))
    display(semantic_examples.head(20))
else:
    print("RUN_SEMANTIC_NEIGHBOR_RETRIEVAL is False; skipping semantic-neighbor retrieval.")

,semantic_recall@10,semantic_recall@50,semantic_mrr,semantic_paper_style_recall_curve_auc,n_text_neighbors,n_queries
0,0.25611,0.486441,0.126576,0.929666,10,2987


,pmid,semantic_positive_pmids,top10_predicted_pmids,hit@10
0,8994101,8994101|10426416|17822430|19926013|20841321|27...,21329758|19349231|18761038|19778619|31470126|3...,False
1,9065511,9065511|15721186|17204823|19948227|20865729|22...,16806986|25278860|20363157|28850297|25309392|2...,False
2,9084599,9084599|11230097|12805308|15193611|16400162|20...,17448452|15901646|18771738|30121445|21718759|1...,False
3,10202567,10202567|12714175|15885517|16837058|21749924|2...,19015080|28425060|33517110|21749924|10202567|2...,True
4,10220229,10220229|12867532|15178381|15590912|18760263|1...,19368829|15459082|15703246|20684660|24675869|1...,True
5,10401985,10401985|12379600|14627638|15275913|17192434|2...,28126038|28523216|17894934|26318935|17977025|9...,True
6,10426416,10426416|17822430|18550759|19595774|20483373|2...,26974440|24888510|25069080|38798104|22644919|1...,True
7,10450889,10450889|10908192|16303149|16387401|18281304|2...,25177286|18281304|25192631|24782735|20815733|3...,True
8,10586971,10586971|11897168|15166105|16055068|20974157|2...,24047380|25788100|31734228|27054402|21172322|2...,True
9,10607401,10607401|10775540|11099442|19465009|19497374|2...,35679333|25844326|23353696|26818506|31377570|2...,False


## 11. Main Baseline Summary Row

In [16]:
summary_row = {
    "model": "pretrained_neurovlm_mlp",
    "n_test_pmids": int(len(test_pmids)),
    "exact_pmid_paper_recall_curve_auc": exact_b2p["paper_recall_curve_auc"],
    "exact_pmid_recall@1": exact_b2p["recall@1"],
    "exact_pmid_recall@5": exact_b2p["recall@5"],
    "exact_pmid_recall@10": exact_b2p["recall@10"],
    "exact_pmid_recall@50": exact_b2p["recall@50"],
    "exact_pmid_mrr": exact_b2p["mrr"],
    "exact_pmid_median_rank": exact_b2p["median_rank"],
}

if "network_metrics" in globals():
    summary_row.update({
        "network_accuracy": network_metrics.get("accuracy"),
        "network_top_2_accuracy": network_metrics.get("top_2_accuracy"),
        "network_macro_auc": network_metrics.get("macro_auc"),
    })
if "mesh_metrics" in globals():
    summary_row.update({
        "mesh_recall@5": mesh_metrics.get("recall@5"),
        "mesh_recall@10": mesh_metrics.get("recall@10"),
        "mesh_map": mesh_metrics.get("map"),
        "mesh_mrr": mesh_metrics.get("mrr"),
        "mesh_median_best_true_term_rank": mesh_metrics.get("median_best_true_term_rank"),
    })
if "semantic_metrics" in globals():
    summary_row.update(semantic_metrics)

summary_df = pd.DataFrame([summary_row])
summary_df.to_csv(OUT_DIR / "main_comparison_summary_row.csv", index=False)
with open(OUT_DIR / "main_comparison_summary_row.json", "w") as f:
    json.dump(summary_row, f, indent=2)
summary_df

,model,n_test_pmids,exact_pmid_paper_recall_curve_auc,exact_pmid_recall@1,exact_pmid_recall@5,exact_pmid_recall@10,exact_pmid_recall@50,exact_pmid_mrr,exact_pmid_median_rank,network_accuracy,...,mesh_recall@10,mesh_map,mesh_mrr,mesh_median_best_true_term_rank,semantic_recall@10,semantic_recall@50,semantic_mrr,semantic_paper_style_recall_curve_auc,n_text_neighbors,n_queries
0,pretrained_neurovlm_mlp,2987,0.833709,0.010378,0.051892,0.09307,0.241379,0.039668,217.0,0.789916,...,0.161159,0.009694,0.074332,167.0,0.25611,0.486441,0.126576,0.929666,10,2987


## 12. Single PMID Sanity Check

In [17]:
SINGLE_PMID = None
TOP_K = 20

if SINGLE_PMID is None:
    query_pos = 0
else:
    matches = np.flatnonzero(test_pmids == str(SINGLE_PMID))
    if len(matches) == 0:
        raise ValueError(f"PMID {SINGLE_PMID} is not in the PubMed test split.")
    query_pos = int(matches[0])

query_pmid = str(test_pmids[query_pos])
query_brain = brain_latent[brain_rows[test_pos[query_pos]]].float().unsqueeze(0)
single_result = nvlm.to_text(query_brain, datasets=["pubmed"], project=True)
single_scores_all = single_result.scores_by_dataset["pubmed"][:, 0]
single_scores_test = single_scores_all[candidate_text_rows]
order = torch.argsort(single_scores_test, descending=True)
true_rank = int((order == query_pos).nonzero(as_tuple=True)[0].item()) + 1

true_row = pub_df.loc[pub_df["pmid"] == query_pmid].head(1).copy()
print("Query/test PMID:", query_pmid)
print("True paper rank among PubMed test candidates:", true_rank, "of", len(test_pmids))
display(true_row.T)

top_positions = order[:TOP_K].cpu().numpy()
top_pmids = test_pmids[top_positions]
top_scores = single_scores_test[top_positions].cpu().numpy()
top_df = pd.DataFrame({
    "rank": np.arange(1, len(top_positions) + 1),
    "pmid": top_pmids,
    "score": top_scores,
    "is_true_paper": top_pmids == query_pmid,
})
meta_cols = [c for c in ["pmid", "title", "name", "abstract", "description"] if c in pub_df.columns]
top_df = top_df.merge(pub_df[meta_cols], on="pmid", how="left")
display(top_df)

Query/test PMID: 8994101
True paper rank among PubMed test candidates: 22 of 2987


,26865
pmid,8994101
pmcid,NaN
doi,None
name,Acoustic neuroma: correlations between morphol...
description,Forty-two patients with acoustic neuroma (AN) ...
train,False
test,True
val,False


,rank,pmid,score,is_true_paper,name,description
0,1,21329758,0.520293,False,A data-driven framework for neural field model...,This paper presents a framework for creating n...
1,2,19349231,0.495528,False,Investigating the benefits of multi-echo EPI f...,Functional MRI studies on humans with BOLD con...
2,3,18761038,0.483327,False,An open-source hardware and software system fo...,Simultaneous recording of electrophysiology an...
3,4,19778619,0.463279,False,Estimating the transfer function from neuronal...,Previous studies using combined electrical and...
4,5,31470126,0.461352,False,SpinDoctor: A MATLAB toolbox for diffusion MRI...,The complex transverse water proton magnetizat...
5,6,34964194,0.450162,False,7T versus 3T MRI in the presurgical evaluation...,MRI has a crucial role in presurgical evaluati...
6,7,22328419,0.434059,False,Investigating causality between interacting br...,"In this work, we investigate the feasibility t..."
7,8,18515149,0.422854,False,Bayesian estimation of synaptic physiology fro...,We describe a Bayesian inference scheme for qu...
8,9,28559192,0.421606,False,Spatio-temporal TGV denoising for ASL perfusio...,In arterial spin labeling (ASL) a perfusion we...
9,10,27693612,0.419230,False,BrainNetCNN: Convolutional neural networks for...,"We propose BrainNetCNN, a convolutional neural..."
